# Cycle 5 - Modèle final : évaluation sur le test, interprétation et sauvegarde

Objectifs de ce cycle :
1. Entraîner la Random Forest avec les hyperparamètres retenus au cycle 4.
2. L'évaluer **une seule fois** sur le jeu de test (2009 à 2013), jamais utilisé jusqu'ici, et la comparer à la baseline naïve sur ces mêmes années.
3. Interpréter le modèle pour le métier : variables importantes, erreur par culture, exemples de recommandations.
4. Réentraîner le modèle sur toutes les données disponibles et l'enregistrer (joblib + métadonnées) pour l'API.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from datetime import datetime as dt
from pathlib import Path

import joblib
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from matplotlib import pyplot as plt
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

import notebooks.utils as utils
# Import par le chemin complet "notebooks.<module>" : le modèle sauvegardé référence cette classe,
# l'API doit pouvoir la ré-importer depuis la racine du projet avec le même chemin.
from notebooks.agritech_feature_engineering import AgritechFeatureEngineering

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Agriculture_optimization")
mlflow.end_run()
mlflow.set_system_metrics_sampling_interval(1)

In [ ]:
dataset = pd.read_csv('../data/rafined/dataset.csv', sep=',')
dataset.drop(columns=['Area ISO Code', 'Area ISO Alpha-2 Code', 'Area ISO Alpha-3 Code'], inplace=True)

## Données d'entrée du modèle

Le modèle n'utilise que les colonnes ci-dessous : ce sont exactement celles que l'API devra lui fournir. Les valeurs climatiques ou de pesticides peuvent être manquantes (`NaN`) : le pipeline les remplace par la médiane, et les indicateurs `missing_*` signalent qu'elles étaient absentes.

In [ ]:
RANDOM_STATE = 42
TARGET = 'hg/ha'

CATEGORICAL_FEATURES = ['Area', 'Item']
NUMERIC_INPUTS = [
    'avg_temp',
    'average_rain_fall_mm_per_year',
    'pesticides_tonnes',
    'pesticides_kg_ha',
    'missing_pesticide_value',
    'missing_rain_value',
    'missing_temp_value',
]
INPUT_FEATURES = CATEGORICAL_FEATURES + NUMERIC_INPUTS

X_train, X_test, y_train, y_test = utils.test_train_time_split(dataset)
X_train = X_train.sort_values('Year')
y_train = y_train.loc[X_train.index]
split_year = int(X_test['Year'].min())

X_train, X_test = X_train[INPUT_FEATURES], X_test[INPUT_FEATURES]

## Modèle final

Random Forest avec les hyperparamètres gagnants du cycle 4 (run MLflow « Cycle 4 - Optimisation Random Forest (RandomizedSearchCV) »), cible transformée en `log1p`, même prétraitement que dans les cycles précédents.

In [ ]:
# Hyperparamètres retenus au cycle 4 (RandomizedSearchCV, meilleure RMSE en validation croisée temporelle)
BEST_PARAMS = {
    'n_estimators': 200,
    'max_features': 0.5,
    'max_depth': 40,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'bootstrap': False,
}


def build_model():
    """Construit le pipeline complet : feature engineering, prétraitement, Random Forest sur cible log1p."""
    numeric_features = NUMERIC_INPUTS + ['aridity', 'avg_temp_square']  # 2 variables créées par AgritechFeatureEngineering

    preprocessor = ColumnTransformer(
        transformers=[
            ('area_encoding',
             TargetEncoder(target_type='continuous', cv=KFold(5, shuffle=True, random_state=RANDOM_STATE)),
             ['Area']),
            ('item_encoding', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Item']),
            ('num_pipeline', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]), numeric_features),
        ],
        remainder='drop'
    )

    pipeline = Pipeline([
        ('features', AgritechFeatureEngineering()),
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(**BEST_PARAMS, random_state=RANDOM_STATE, n_jobs=-1)),
    ])

    return TransformedTargetRegressor(regressor=pipeline, func=np.log1p, inverse_func=np.expm1)

## Évaluation finale sur le jeu de test (2009 à 2013)

Le modèle est entraîné sur 1990 à 2008, puis évalué une seule fois sur 2009 à 2013. La baseline naïve (médiane pays × culture) est recalculée sur exactement les mêmes années, ce qui rend la comparaison directe.

En plus de R², MAE et RMSE, on mesure une **précision économique** : la part des prédictions qui tombent à ±15 % du rendement réel. C'est l'indicateur le plus parlant pour un agriculteur : « dans combien de cas la prévision est-elle assez fiable pour décider ? ».

In [ ]:
def precision_15pct(y_true, y_pred):
    """Part des prédictions à ±15 % du rendement réel."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.mean(np.abs(y_pred - y_true) <= 0.15 * y_true))


# Modèle entraîné sur le passé (1990-2008) uniquement
eval_model = build_model().fit(X_train, y_train)
pred_test = eval_model.predict(X_test)

test_metrics = utils.evaluate(y_test, pred_test)
test_metrics['precision_15pct'] = precision_15pct(y_test, pred_test)

# Baseline naïve sur les mêmes années : médiane historique du couple (pays, culture)
train_df = X_train.assign(**{TARGET: y_train})
mediane_pays_culture = train_df.groupby(['Area', 'Item'])[TARGET].median()
mediane_culture = train_df.groupby('Item')[TARGET].median()
pred_baseline = X_test.join(mediane_pays_culture.rename('pred'), on=['Area', 'Item'])['pred']
pred_baseline = pred_baseline.fillna(X_test['Item'].map(mediane_culture))

baseline_metrics = utils.evaluate(y_test, pred_baseline)
baseline_metrics['precision_15pct'] = precision_15pct(y_test, pred_baseline)

comparaison_test = pd.DataFrame(
    {'Baseline naïve': baseline_metrics, 'Random Forest optimisée': test_metrics}
).T[['r2', 'mae', 'rmse', 'precision_15pct']]
comparaison_test.loc['Évolution (%)'] = (
    (comparaison_test.loc['Random Forest optimisée'] / comparaison_test.loc['Baseline naïve'] - 1) * 100
)
comparaison_test.round(3)

In [ ]:
# Erreur par culture sur le test : le modèle est-il fiable pour toutes les cultures ?
erreurs = pd.DataFrame({'Item': X_test['Item'], 'reel': y_test, 'modele': pred_test, 'baseline': pred_baseline})

par_culture = erreurs.groupby('Item').apply(lambda g: pd.Series({
    'rendement_median': g['reel'].median(),
    'mae_modele': (g['modele'] - g['reel']).abs().mean(),
    'mae_baseline': (g['baseline'] - g['reel']).abs().mean(),
    'erreur_relative_modele_%': ((g['modele'] - g['reel']).abs() / g['reel'].clip(lower=1)).median() * 100,
    'precision_15pct_modele': precision_15pct(g['reel'], g['modele']),
}), include_groups=False)
par_culture['gain_vs_baseline_%'] = (par_culture['mae_modele'] / par_culture['mae_baseline'] - 1) * 100
par_culture.sort_values('rendement_median').round(2)

## Variables les plus importantes

In [ ]:
fitted_pipeline = eval_model.regressor_
rf = fitted_pipeline.named_steps['model']
noms_variables = fitted_pipeline.named_steps['preprocessor'].get_feature_names_out()

df_importance = pd.DataFrame({'Variable': noms_variables, 'Importance': rf.feature_importances_})
df_importance['Variable'] = df_importance['Variable'].str.split('__').str[1]

# Regroupement des 10 colonnes one-hot de la culture en une seule variable
df_importance.loc[df_importance['Variable'].str.startswith('Item_'), 'Variable'] = 'Item'
df_importance = df_importance.groupby('Variable', as_index=False)['Importance'].sum()

libelles = {
    'Item': 'Culture',
    'Area': 'Pays',
    'pesticides_kg_ha': 'Pesticides (kg/ha)',
    'pesticides_tonnes': 'Pesticides (tonnes, total pays)',
    'average_rain_fall_mm_per_year': 'Pluviométrie annuelle',
    'avg_temp': 'Température moyenne',
    'avg_temp_square': 'Température² (effet non linéaire)',
    'aridity': "Indice d'aridité",
    'missing_temp_value': 'Température manquante',
    'missing_rain_value': 'Pluviométrie manquante',
    'missing_pesticide_value': 'Pesticides manquants',
}
df_importance['Variable'] = df_importance['Variable'].map(libelles).fillna(df_importance['Variable'])
df_importance = df_importance.sort_values('Importance', ascending=False)

fig_importance, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=df_importance, x='Importance', y='Variable', color='#2a78d6', ax=ax)
ax.set_title("Importance des variables dans la prédiction du rendement", loc='left')
ax.set_xlabel("Importance (part de la réduction d'erreur)")
ax.set_ylabel("")
fig_importance.tight_layout()
plt.show()

df_importance.round(3)

## Exemples de recommandations

Logique de l'endpoint `/recommend` : pour un contexte donné (pays, climat, pesticides), on prédit le rendement de chacune des 10 cultures et on les trie par rendement décroissant.

Le rendement brut favorise mécaniquement les tubercules (une pomme de terre pèse beaucoup plus lourd par hectare qu'une céréale). On ajoute donc un **indice relatif** : le rendement prédit divisé par le rendement médian mondial de la culture. Un indice supérieur à 1 signifie que ce contexte est plus favorable que la moyenne pour cette culture.

In [ ]:
CULTURES = sorted(dataset['Item'].unique())
mediane_mondiale = dataset.groupby('Item')[TARGET].median()


def recommend(model, context):
    """Prédit le rendement de chaque culture pour un contexte, trié par rendement décroissant."""
    rows = pd.DataFrame([{**context, 'Item': item} for item in CULTURES])[INPUT_FEATURES]
    result = pd.DataFrame({'Culture': CULTURES, 'rendement_predit_hg_ha': model.predict(rows)})
    result['indice_vs_mediane_mondiale'] = result['rendement_predit_hg_ha'] / result['Culture'].map(mediane_mondiale).values
    return result.sort_values('rendement_predit_hg_ha', ascending=False).reset_index(drop=True)


def context_from_data(area, year):
    """Contexte réel (climat, pesticides) d'un pays pour une année du dataset."""
    row = dataset[(dataset['Area'] == area) & (dataset['Year'] == year)].iloc[0]
    return row[[c for c in INPUT_FEATURES if c != 'Item']].to_dict()


exemples = {}
for pays in ['France', 'India', 'Kenya']:
    reco = recommend(eval_model, context_from_data(pays, 2013))
    cultures_pratiquees = set(dataset.loc[dataset['Area'] == pays, 'Item'])
    reco['cultivee_dans_le_pays'] = reco['Culture'].isin(cultures_pratiquees)
    exemples[pays] = reco
    print(f"--- {pays} (contexte 2013) ---")
    display(reco.round(2))

## Modèle de production : réentraînement sur toutes les données

L'évaluation est terminée et elle a été faite honnêtement sur des années jamais vues. Pour l'API, on réentraîne le même modèle (mêmes hyperparamètres) sur **toutes** les années disponibles (1990 à 2013) : il bénéficie ainsi des données les plus récentes, ce qui compte puisque les rendements progressent dans le temps. Les performances annoncées restent celles mesurées sur le test ci-dessus.

In [ ]:
X_all = dataset[INPUT_FEATURES]
y_all = dataset[TARGET]

final_model = build_model().fit(X_all, y_all)

## Sauvegarde pour l'API

- `models/agriculture_model.joblib` : le pipeline complet (feature engineering, prétraitement, modèle, transformation inverse de la cible). Il prend en entrée un DataFrame avec les colonnes `INPUT_FEATURES` et renvoie directement un rendement en hg/ha.
- `models/agriculture_model_metadata.json` : tout ce dont l'API a besoin pour valider les entrées et construire `/recommend` (colonnes attendues, liste des cultures et des pays connus, performances).

In [ ]:
models_dir = Path('..').resolve() / 'models'
models_dir.mkdir(exist_ok=True)
model_path = models_dir / 'agriculture_model.joblib'
metadata_path = models_dir / 'agriculture_model_metadata.json'

version = dt.now().strftime("%Y%m%d_%H%M%S")

metadata = {
    'version': version,
    'created_at': dt.now().isoformat(timespec='seconds'),
    'sklearn_version': sklearn.__version__,
    'model': 'RandomForestRegressor (cible log1p)',
    'hyperparameters': BEST_PARAMS,
    'random_state': RANDOM_STATE,
    'target': TARGET,
    'target_unit': 'hectogrammes par hectare (hg/ha) ; 10 000 hg/ha = 1 t/ha',
    'input_features': INPUT_FEATURES,
    'input_dtypes': {c: str(X_all[c].dtype) for c in INPUT_FEATURES},
    'optional_features': [c for c in NUMERIC_INPUTS if not c.startswith('missing_')],
    'cultures': CULTURES,
    'pays': sorted(dataset['Area'].unique()),
    # Bornes observées des variables numériques (utilisées par le dashboard pour les sliders)
    'feature_ranges': {
        c: {
            'min': round(float(dataset[c].min()), 2),
            'median': round(float(dataset[c].median()), 2),
            'max': round(float(dataset[c].max()), 2),
        }
        for c in NUMERIC_INPUTS if not c.startswith('missing_')
    },
    'training_years': [int(dataset['Year'].min()), int(dataset['Year'].max())],
    'evaluation': {
        'methode': f"entraîné sur {int(dataset['Year'].min())}-{split_year - 1}, évalué sur {split_year}-{int(dataset['Year'].max())}",
        'modele': {k: round(float(v), 4) for k, v in test_metrics.items()},
        'baseline_mediane_pays_culture': {k: round(float(v), 4) for k, v in baseline_metrics.items()},
    },
}

# compress=3 : fichier environ 4 fois plus léger (720 Mo -> 180 Mo), chargement aussi rapide
joblib.dump(final_model, model_path, compress=3)
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"Modèle enregistré : {model_path} ({model_path.stat().st_size / 1e6:.1f} Mo)")
print(f"Métadonnées enregistrées : {metadata_path}")

In [ ]:
# Vérification : le modèle rechargé donne exactement les mêmes prédictions
reloaded = joblib.load(model_path)
sample = X_all.sample(5, random_state=RANDOM_STATE)
assert np.allclose(reloaded.predict(sample), final_model.predict(sample))

# Exemple d'appel tel que l'API le fera : une ligne, avec une valeur manquante
exemple_api = pd.DataFrame([{
    'Area': 'France', 'Item': 'Blé',
    'avg_temp': 12.5, 'average_rain_fall_mm_per_year': 867.0,
    'pesticides_tonnes': None, 'pesticides_kg_ha': None,
    'missing_pesticide_value': 1, 'missing_rain_value': 0, 'missing_temp_value': 0,
}])[INPUT_FEATURES]
print(f"Rendement prédit pour le blé en France : {reloaded.predict(exemple_api)[0]:.0f} hg/ha")

## Enregistrement dans MLflow

In [ ]:
with mlflow.start_run(run_name="Cycle 5 - Modèle final (évaluation test)", log_system_metrics=True):

    mlflow.set_tags({
        'date': dt.now().strftime("%Y-%m-%d"),
        'cycle': '5_modele_final',
        'model_type': 'random_forest',
        'model_version': version,
        'mlflow.note.content': (
            "Modèle final : Random Forest avec les hyperparamètres du cycle 4, cible log1p. "
            f"Évalué une seule fois sur le test ({split_year}-{int(dataset['Year'].max())}) et comparé à la baseline "
            "naïve sur les mêmes années. Le modèle sauvegardé pour l'API est réentraîné sur toutes les années."
        ),
    })

    mlflow.log_params({
        'split': 'temporel',
        'split_year': split_year,
        'model': 'RandomForestRegressor',
        'target_transform': 'log1p',
        'features': ', '.join(INPUT_FEATURES),
        'random_state': RANDOM_STATE,
        **{f'model__{k}': v for k, v in BEST_PARAMS.items()},
    })

    mlflow.log_metrics({
        **{f'test_{k}': v for k, v in test_metrics.items()},
        **{f'baseline_test_{k}': v for k, v in baseline_metrics.items()},
        'gain_rmse_vs_baseline_pct': (test_metrics['rmse'] / baseline_metrics['rmse'] - 1) * 100,
        'gain_mae_vs_baseline_pct': (test_metrics['mae'] / baseline_metrics['mae'] - 1) * 100,
    })

    mlflow.log_figure(fig_importance, 'importance_variables.png')
    mlflow.log_table(par_culture.reset_index(), artifact_file='erreur_par_culture.json')
    for pays, reco in exemples.items():
        mlflow.log_table(reco, artifact_file=f'recommandations/{pays}.json')
    mlflow.log_artifact(str(model_path), artifact_path='modele')
    mlflow.log_artifact(str(metadata_path), artifact_path='modele')